In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import plotly.express as px

In [2]:
def calibrate(df, ws='WS', v='V', t='T'):
    """
    Ajusta la ecuación de calibración del sensor a partir de un DataFrame.
    
    Parámetros:
        df (DataFrame): DataFrame con los datos.
        ws (str): Nombre de la columna que contiene la velocidad del viento (WS).
        v (str): Nombre de la columna con el voltaje ajustado (V).
        t (str): Nombre de la columna con la temperatura (T).
    
    Retorna:
        dict: Un diccionario con los siguientes elementos:
            - 'a': Factor multiplicativo (a = exp(intercepto)).
            - 'b': Exponente asociado al voltaje ajustado.
            - 'c': Exponente asociado a la temperatura.
            - 'r2': Coeficiente de determinación.
            - 'equation': Cadena de texto con la ecuación calibrada en la forma:
                          WS = a * V_adj^b * T^c
    """
    
    # Filtrar para evitar valores no válidos (asegurarse de que WS, V_adj y T sean mayores a 0)
    df_filtrado = df[(df[ws] > 0) & (df[v] > 0) & (df[t] > 0)]
    
    # Transformar las variables mediante el logaritmo natural
    # Nota: se usa doble corchete para mantener X como array 2D
    X = np.log(df_filtrado[[v, t]])
    y = np.log(df_filtrado[ws])
    
    # Ajustar el modelo de regresión lineal (en dominio logarítmico)
    modelo = LinearRegression()
    modelo.fit(X, y)
    
    # Obtener el intercepto y los coeficientes
    intercepto = modelo.intercept_      # ln(a)
    coeficientes = modelo.coef_           # [b, c]
    
    # Convertir el intercepto a a (ya que ln(a) = intercepto => a = exp(intercepto))
    a = np.exp(intercepto)
    b = coeficientes[0]
    c = coeficientes[1]
    
    # Crear la ecuación calibrada en formato de texto
    equation = f"WS = {a:.4f} * {v}^({b:.4f}) * {t}^({c:.4f})"
    
    # Calcular el coeficiente de determinación R^2
    r2 = modelo.score(X, y)
        
    # Retornar los parámetros en un diccionario
    return {
        "a": a,
        "b": b,
        "c": c,
        "r2": r2,
        "equation": equation
    }

## Voltaje con 0 m/s

In [3]:
f = "../data/zero.csv"
zero = pd.read_csv(f, index_col=0, parse_dates=True)
zero

,voltage,temp
ts,,
2025-02-06 16:15:00,1.1426,28.32
2025-02-06 16:15:02,1.1426,28.32
2025-02-06 16:15:04,1.1475,28.57
2025-02-06 16:15:06,1.1475,28.82
2025-02-06 16:15:08,1.1426,28.57
...,...,...
2025-02-06 17:14:52,1.1572,32.32
2025-02-06 17:14:54,1.1621,32.32
2025-02-06 17:14:56,1.1621,32.57


In [4]:
fig = px.scatter(zero, x=zero.index, y=zero.voltage)
fig.show()

In [5]:
zero.voltage.mean()

np.float64(1.1621905660377356)

Tras 1 hora de mediciones a una velocidad de 0 m/s, se ha encontrado el voltaje del WindSensor a dicho valor, el cual corresponde a **1.1621 V**

## Voltaje de WindSensor

In [6]:
f = "../data/windsensor.csv"
voltage = pd.read_csv(f, index_col=0, parse_dates=True)
voltage = voltage.between_time("09:30:00","10:30:00")
voltage = voltage.resample("1s").mean()
voltage["time"] = voltage.index.diff()
voltage.time = voltage.time.dt.total_seconds()

In [7]:
voltage

,voltage,temp,time
ts,,,
2025-02-05 09:30:00,1.3086,25.81,NaN
2025-02-05 09:30:01,1.4062,25.81,1.0
2025-02-05 09:30:02,1.3965,26.06,1.0
2025-02-05 09:30:03,1.4355,26.56,1.0
2025-02-05 09:30:04,1.3770,25.56,1.0
...,...,...,...
2025-02-05 10:29:56,1.3965,26.06,1.0
2025-02-05 10:29:57,1.3770,25.81,1.0
2025-02-05 10:29:58,1.3721,26.06,1.0


## WS de anemometro

In [8]:
f = "../data/windspeed.csv"
anemometro = pd.read_csv(f, index_col=0, parse_dates=True)

In [9]:
anemometro

,u,v,w,ws
fecha,,,,
2025-02-05 09:30:00,-0.03,0.02,0.06,0.0700
2025-02-05 09:30:01,-0.17,0.09,0.06,0.2015
2025-02-05 09:30:02,-0.24,0.15,0.08,0.2941
2025-02-05 09:30:03,-0.13,0.00,0.07,0.1476
2025-02-05 09:30:04,-0.13,-0.08,0.09,0.1772
...,...,...,...,...
2025-02-05 10:29:56,0.10,-0.23,-0.08,0.2632
2025-02-05 10:29:57,0.01,-0.18,-0.19,0.2619
2025-02-05 10:29:58,-0.07,-0.18,-0.22,0.2927


## WS vs. Voltage

In [10]:
data = pd.concat([anemometro[['ws']], voltage[['voltage', 'temp']]], axis=1)
data

,ws,voltage,temp
2025-02-05 09:30:00,0.0700,1.3086,25.81
2025-02-05 09:30:01,0.2015,1.4062,25.81
2025-02-05 09:30:02,0.2941,1.3965,26.06
2025-02-05 09:30:03,0.1476,1.4355,26.56
2025-02-05 09:30:04,0.1772,1.3770,25.56
...,...,...,...
2025-02-05 10:29:56,0.2632,1.3965,26.06
2025-02-05 10:29:57,0.2619,1.3770,25.81
2025-02-05 10:29:58,0.2927,1.3721,26.06
2025-02-05 10:29:59,0.3076,1.3867,26.31


In [11]:
fig = px.scatter(data, x=data.index, y=['voltage','ws'])
fig.show()

Se aplica resta de **1.1621 V** al voltaje obtenido en la campaña.

In [12]:
data.voltage = data.voltage - 1.1621

In [13]:
fig = px.scatter(data, x=data.voltage, y=data.ws)
fig.show()

### Correlación con resampleo de 3s

In [14]:
tre = data.resample("3s").mean()
px.scatter(tre, x=tre.voltage, y=tre.ws)

In [15]:
calibrate(tre, ws='ws', v='voltage', t='temp')

{'a': np.float64(9.857185418106017),
 'b': np.float64(1.4003223195579135),
 'c': np.float64(-0.4789448443202256),
 'r2': 0.7622174894564582,
 'equation': 'WS = 9.8572 * voltage^(1.4003) * temp^(-0.4789)'}

### Correlación con resampleo de 5s

In [16]:
cinq = data.resample("5s").mean()
px.scatter(cinq, x=cinq.voltage, y=cinq.ws)

In [17]:
calibrate(cinq, ws='ws', v='voltage', t='temp')

{'a': np.float64(9.939778763174985),
 'b': np.float64(1.4109201497954562),
 'c': np.float64(-0.4739215146328439),
 'r2': 0.807940471288248,
 'equation': 'WS = 9.9398 * voltage^(1.4109) * temp^(-0.4739)'}

### Correlación con resampleo de 10s

In [18]:
dix = data.resample("10s").mean()
px.scatter(dix, x=dix.voltage, y=dix.ws)

In [19]:
calibrate(dix, ws='ws', v='voltage', t='temp')

{'a': np.float64(26.343118785192097),
 'b': np.float64(1.4272992243818632),
 'c': np.float64(-0.7631033960198768),
 'r2': 0.8636922725931118,
 'equation': 'WS = 26.3431 * voltage^(1.4273) * temp^(-0.7631)'}